# Lab 16 - When to Use an Agent: The Decision Classifier
**Section covered: 06 (When to Use an Agent and When Not To)**

---

## What this lab builds

Section 06 gave you a decision framework with four quadrants:
- YES: dynamic conversation, multi-step multi-tool, unstructured input
- NO: simple rule-based predictable tasks

This lab builds a **task classifier** that applies that framework to any task description
and returns a structured recommendation: agent, workflow, or single LLM call.

You will classify 10 real tasks and see where each one lands, then see the
cost comparison to understand why getting this decision right matters.

---

## The core insight

Agents are not always the answer. Every time you choose an agent over a workflow,
you are accepting: more complexity, more latency, more cost, more failure modes.

That trade-off is worth making when the task genuinely needs dynamic decision-making.
It is not worth making when a simple if-then-else handles it perfectly.

This classifier helps you make that call systematically.

## Step 1 - Setup

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()
print("Client ready")
print()
print("This lab builds a decision classifier that analyses tasks and")
print("recommends whether to use an agent or a simpler approach.")
print("It applies the decision framework from Section 06 of the bootcamp.")

## Step 2 - The Decision Framework

The four quadrants from Section 06 as code.

In [ ]:
# ===================================================================
# THE DECISION FRAMEWORK FROM SECTION 06
# Three signals that mean YES - one clear signal that means NO
# ===================================================================

DECISION_FRAMEWORK = {
    "yes_signals": {
        "dynamic_conversation": (
            "Task needs context across multiple turns. "
            "User will ask follow-up questions or switch topics mid-conversation. "
            "The flow cannot be predicted in advance."
        ),
        "multi_step_multi_tool": (
            "Task requires querying multiple data sources or systems. "
            "Steps depend on each other (step 2 needs the result of step 1). "
            "Requires both reading from AND writing to real systems."
        ),
        "unstructured_input": (
            "Input is free-form text, documents, images, or audio. "
            "Traditional software cannot parse this reliably. "
            "Content varies significantly from case to case."
        ),
    },
    "no_signals": {
        "rule_based": (
            "Task is straightforward and fully predictable. "
            "Same input always produces same output. "
            "Could be written as a simple if-then-else workflow. "
            "Happens infrequently or does not justify autonomous execution."
        )
    }
}

def show_framework():
    print("DECISION FRAMEWORK - when to use an agent")
    print("=" * 55)
    print()
    print("YES - use an agent if the task has ANY of these:")
    for name, desc in DECISION_FRAMEWORK["yes_signals"].items():
        print(f"  [{name}]")
        print(f"   {desc[:80]}")
        print()
    print("NO - do NOT use an agent if:")
    for name, desc in DECISION_FRAMEWORK["no_signals"].items():
        print(f"  [{name}]")
        print(f"   {desc[:80]}")

show_framework()

## Step 3 - The Task Classifier

An LLM that analyses tasks against the framework and returns structured recommendations.

In [ ]:
# ===================================================================
# THE TASK CLASSIFIER
# An LLM that analyses a task against the decision framework
# and returns a structured recommendation
# ===================================================================

CLASSIFIER_SYSTEM = """You are a software architecture advisor specialising in AI systems.

Your job is to analyse a task description and recommend whether it needs an agent
or a simpler approach. Apply this decision framework strictly:

USE AN AGENT if the task has any of:
  dynamic_conversation: needs context across turns, unpredictable flow, multi-topic
  multi_step_multi_tool: multiple tools/systems, sequential steps, real-world actions
  unstructured_input: free-form text/documents/images, variable content structure

DO NOT USE AN AGENT if:
  rule_based: straightforward, predictable, could be an if-then workflow, infrequent

Return ONLY a JSON object with this exact structure:
{
  "recommendation": "agent" or "workflow" or "single_llm_call",
  "confidence": "high" or "medium" or "low",
  "primary_signal": "dynamic_conversation" or "multi_step_multi_tool" or "unstructured_input" or "rule_based",
  "reasoning": "one sentence explaining the recommendation",
  "what_to_build": "one specific sentence describing the simplest implementation"
}"""

def classify_task(task_description: str) -> dict:
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=512,
        system=CLASSIFIER_SYSTEM,
        messages=[{"role": "user", "content": f"Analyse this task:\n\n{task_description}"}]
    )
    raw = response.content[0].text.strip()
    if "```" in raw:
        parts = raw.split("```")
        raw = parts[1] if len(parts) > 1 else parts[0]
        if raw.startswith("json"):
            raw = raw[4:]
    try:
        return json.loads(raw.strip())
    except json.JSONDecodeError:
        return {"recommendation": "unknown", "reasoning": raw[:100]}

def analyse_and_display(task: str):
    print(f"Task: {task[:80]}...")
    result = classify_task(task)
    rec    = result.get("recommendation", "?")
    signal = result.get("primary_signal", "?")
    conf   = result.get("confidence", "?")
    reason = result.get("reasoning", "")
    build  = result.get("what_to_build", "")

    icon = "AGENT" if rec == "agent" else ("WORKFLOW" if rec == "workflow" else "SINGLE CALL")
    print(f"  Recommendation: [{icon}]  Signal: {signal}  Confidence: {conf}")
    print(f"  Why: {reason}")
    print(f"  Build: {build}")
    print()
    return result

print("Task classifier ready")

## Step 4 - Classify 10 Tasks

Spanning all four quadrants. Watch where each one lands.

In [ ]:
# ===================================================================
# CLASSIFY 10 TASKS - covering all four quadrants of the framework
# ===================================================================

tasks = [
    # Clear agent cases
    "A customer contacts our support chat. They want to check their order status, "
    "change their delivery address, and ask about our returns policy - all in one conversation.",

    "An analyst asks: pull last month sales data from our database, "
    "generate a bar chart of top 10 products, and email it to the leadership team.",

    "Process a folder of 500 scanned invoices (PDF and images) and extract "
    "vendor name, invoice date, and total amount from each one.",

    # Clear non-agent cases
    "When a customer submits a refund request, check if it is within 30 days, "
    "and if yes issue the refund automatically. This happens 50 times a day.",

    "Every night at 2am, back up the production database to S3.",

    # Edge cases - interesting to see where it lands
    "Answer the question: what is the capital of France?",

    "Read a user message and decide if it is positive, negative, or neutral sentiment. "
    "We do this for 10,000 messages per hour.",

    "A user uploads a photo of their receipt and asks for the total amount to be "
    "reimbursed in their company expense account.",

    "Generate a weekly email summary of all open support tickets, "
    "categorised by priority and team.",

    "A new employee needs their laptop configured: install approved software, "
    "set up VPN access, add them to the correct security groups, "
    "and send them a welcome email with their credentials.",
]

print("Classifying 10 tasks against the Section 06 decision framework...")
print("=" * 60)

results = []
for task in tasks:
    result = analyse_and_display(task)
    results.append(result)

In [ ]:
# Summary of classification results
print("=" * 60)
print("CLASSIFICATION SUMMARY")
print("=" * 60)

recommendations = [r.get("recommendation", "?") for r in results]
signals         = [r.get("primary_signal", "?") for r in results]

print(f"Agent recommended:          {recommendations.count('agent')}/{len(results)}")
print(f"Workflow recommended:       {recommendations.count('workflow')}/{len(results)}")
print(f"Single LLM call:            {recommendations.count('single_llm_call')}/{len(results)}")
print()
print("Primary signals triggered:")
from collections import Counter
signal_counts = Counter(signals)
for signal, count in signal_counts.most_common():
    print(f"  {signal}: {count}")

print()
print("Key lesson from Section 06:")
print("  Not every task needs an agent. Agents add complexity, cost,")
print("  and failure points. Use them only when the task genuinely")
print("  requires dynamic decision-making that cannot be hardcoded.")

## Step 5 - Cost Comparison: Why the Build Decision Matters

In [ ]:
# ===================================================================
# COST COMPARISON: agent vs single call vs workflow
# This shows WHY the build-right-level-of-complexity decision matters
# ===================================================================

print("COST COMPARISON - same task at different complexity levels")
print("=" * 60)
print()

# Simulate costs for "check if refund is within 30 days"
task_description = "Check if a refund request is within the 30-day window and issue it."
print(f"Task: {task_description}")
print()

costs = {
    "Hardcoded workflow (if/else + API call)": {
        "tokens_per_call": 0,
        "latency_ms": 50,
        "cost_per_1000_calls": 0.00,
        "failure_modes": "API downtime",
        "best_for": "this task"
    },
    "Single LLM call (parse + decide)": {
        "tokens_per_call": 150,
        "latency_ms": 800,
        "cost_per_1000_calls": 0.45,
        "failure_modes": "hallucination, inconsistency",
        "best_for": "unstructured input parsing"
    },
    "Full agent (loop + tools)": {
        "tokens_per_call": 1500,
        "latency_ms": 4000,
        "cost_per_1000_calls": 4.50,
        "failure_modes": "multiple failure points, latency variance",
        "best_for": "multi-step, multi-tool, dynamic tasks"
    }
}

for approach, data in costs.items():
    print(f"  [{approach}]")
    print(f"    Tokens/call:       {data['tokens_per_call']}")
    print(f"    Latency:           ~{data['latency_ms']}ms")
    print(f"    Cost per 1K calls: ${data['cost_per_1000_calls']:.2f}")
    print(f"    Failure modes:     {data['failure_modes']}")
    print(f"    Best for:          {data['best_for']}")
    print()

print("At 50K calls/day (a busy support system):")
print("  Workflow:        $0/day          $0/month")
print("  Single LLM call: $22.50/day     $675/month")
print("  Agent:           $225/day        $6,750/month")
print()
print("Choosing the right level of complexity is an engineering and cost decision.")
print("Use the Section 06 framework every time before reaching for an agent.")

## Key Takeaways

| Signal | Means | Build |
|--------|-------|-------|
| Dynamic conversation | Context across turns, unpredictable flow | Agent |
| Multi-step multi-tool | Sequential steps, multiple systems | Agent |
| Unstructured input | Free-form text/docs/images | Agent |
| Rule-based, predictable | Same path every time | Workflow or single call |

The cost multiplier is real: an agent processing 50K calls/day costs 10-30x more
than a workflow doing the same job. The decision framework from Section 06
is not just architecture advice - it is cost engineering.

---
This completes the full lab series. All 16 labs map directly to sections in the bootcamp HTML.